## Assignment 1: Text Representation and Semantic Modeling

### Assignment Overview

This assignment covers the fundamental concepts of text preprocessing, feature extraction, and word embeddings discussed in the first three weeks of lectures. The goal is to build a solid practical foundation in preparing text data for downstream machine learning tasks and to understand how semantic meaning can be captured in vector representations.

You are required to complete five coding-related tasks. For each task, you will be working with a specified dataset. Please submit your solutions in this single Jupyter Notebook (`.ipynb`) file, clearly marking each task. Ensure your code is well-commented and your findings are explained in markdown cells where requested.

### Task 1: Advanced Text Cleaning and Normalization (20 Marks)

**Objective:** To clean and normalize a dataset of noisy, real-world text by implementing a comprehensive preprocessing pipeline.

**Description:** Real-world text is messy. In this task, you will write a single, reusable Python function named `clean_text(raw_text)` that performs a series of preprocessing steps on a raw text string.

**Your `clean_text` function must perform the following steps in order:**

1. Convert the input text to lowercase.
2. Remove any URLs (e.g., `http://...`, `https://...`).
3. Remove any Twitter handles (e.g., `@username`).
4. Remove any hashtags (e.g., `#topic`).
5. Remove all punctuation and special characters, leaving only letters, numbers, and spaces.
6. Remove common English stop words (e.g., "the", "a", "is").
7. Perform lemmatization on the remaining words to reduce them to their base form (e.g., "running" becomes "run").

**Your task is to:**

* Apply this function to the 'text' column of the Twitter US Airline Sentiment dataset.
* Display the head of a new DataFrame containing two columns: the original, raw tweet and the corresponding cleaned text.

**Dataset:**

* **Twitter US Airline Sentiment Dataset:** This dataset contains tweets about major U.S. airlines and is ideal for this task due to its noisy nature.
* **Access:** You can download it from Kaggle: [Twitter US Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment)

In [21]:
# Your code for Task 1 here
# Import necessary libraries (e.g., pandas, re, nltk)
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

def clean_text(raw_text):
    # Implement the 7 steps of the cleaning pipeline
    if pd.isna(raw_text): 
        return ""
        
    # 1. 转换为小写
    text = raw_text.lower()
    
    # 2. 移除URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. 移除用户名
    text = re.sub(r'@\w+', '', text)
    
    # 4. 移除标签
    text = re.sub(r'#\w+', '', text)
    
    # 5. 移除标点符号和特殊字符
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # 6. 移除常见的英文停用词
    stop_words = set(stopwords.words('english'))
    # 分词
    tokens = word_tokenize(text)
    # 过滤停用词
    filtered_tokens = [token for token in tokens if token not in stop_words]
    
    # 7. 对剩余的词进行词形还原
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    
    # 重新组成字符串
    cleaned_text = ' '.join(lemmatized_tokens)
    
    # 移除多余空格
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text


# Load the dataset
df = pd.read_csv('Tweets.csv')

# Apply the function and display the result
df['cleaned_text'] = df['text'].apply(clean_text)
print("\n原始文本 vs 清洗后文本:")
print(df[['text', 'cleaned_text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



原始文本 vs 清洗后文本:
                                                text  \
0                @VirginAmerica What @dhepburn said.   
1  @VirginAmerica plus you've added commercials t...   
2  @VirginAmerica I didn't today... Must mean I n...   
3  @VirginAmerica it's really aggressive to blast...   
4  @VirginAmerica and it's a really big bad thing...   

                                        cleaned_text  
0                                               said  
1             plus added commercial experience tacky  
2             today must mean need take another trip  
3  really aggressive blast obnoxious entertainmen...  
4                               really big bad thing  


### Task 2: Implementing a Byte Pair Encoding (BPE) Tokenizer (20 Marks)

**Objective:** To understand the mechanics of subword tokenization by implementing a simplified BPE algorithm from scratch.

**Description:** Subword tokenization offers a robust solution for handling rare words and large vocabularies. You will implement the core logic of the BPE algorithm.

**Your task is to:**

1. Implement a training function, `train_bpe(corpus, num_merges)`, that takes the cleaned text from Task 1 and the number of merges as input.
2. Inside `train_bpe`, initialize a vocabulary with all unique individual characters from the corpus.
3. Implement a loop that runs for `num_merges` iterations. In each iteration, it should:
   * Find the most frequent adjacent pair of symbols in the current representation of the corpus.
   * Merge this pair into a new, single symbol.
   * Add the new symbol to a list of merge rules.
   * Replace all occurrences of the original pair in the corpus with the new merged symbol.
4. Train your BPE tokenizer on the 'cleaned_text' from Task 1. Set the number of merge operations to 500.
5. After training, save the final learned vocabulary (all unique tokens, including initial characters and merged symbols) to a text file named `bpe_vocab.txt`, with one token per line.
6. Implement a separate tokenization function, `tokenize(sentence, merge_rules)`, that uses the learned merge rules to tokenize the fixed sentence: `"this is a new flight"`. Show the resulting list of tokens.

**Dataset:**

* Use the cleaned text data you generated in Task 1 from the **Twitter US Airline Sentiment Dataset**.

In [22]:
# Your code for Task 2 here
from collections import defaultdict, Counter
import re

# Get the cleaned corpus from Task 1
# cleaned_corpus = ... 

# Train the BPE tokenizer
# vocab, merge_rules = train_bpe(cleaned_corpus, 500)

# Save the vocabulary

# Tokenize the sample sentence

def get_pair_stats(vocab):
    pairs = defaultdict(int)
    # 每个单词及频率
    for word, freq in vocab.items():
        # 分割成符号列表
        symbols = word.split()
        # 查相邻符号对
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            # 频率累加
            pairs[pair] += freq
    return pairs

def merge_vocab_once(pair_to_merge, vocab_in):
    vocab_out = {}
    # 转换为字符串
    bigram_str = ' '.join(pair_to_merge)
    # 合并为新符号
    new_symbol = ''.join(pair_to_merge)

    for word, freq in vocab_in.items():
        # 全部替换为新符号
        new_word = word.replace(bigram_str, new_symbol)
        # 更新后重新存入词汇表
        vocab_out[new_word] = freq

    return vocab_out

def train_bpe(corpus, num_merges):
    # 预处理语料库
    processed_words_list = []
    initial_symbols = set() 

    for sentence in corpus:
        if not sentence.strip():
            continue
        words = sentence.strip().split()
        for word in words:
            if word:
                processed_word = ' '.join(list(word)) + ' </w>'
                processed_words_list.append(processed_word)
                initial_symbols.update(list(word))
                initial_symbols.add('</w>')

    # 初始化词汇表和返回值
    vocab = Counter(processed_words_list)

    merge_rules = []
    final_vocabulary = set(initial_symbols)

    # 合并
    for i in range(num_merges):
        pair_stats = get_pair_stats(vocab)

        if not pair_stats:
            print(f"提前停止: 第 {i} 次迭代时未找到符号对")
            break

        most_frequent_pair = max(pair_stats, key=pair_stats.get)
        new_symbol = ''.join(most_frequent_pair)
        final_vocabulary.add(new_symbol)

        merge_rules.append(most_frequent_pair)

        vocab = merge_vocab_once(most_frequent_pair, vocab)

    return final_vocabulary, merge_rules

def tokenize(sentence, merge_rules):
    # 预处理句子
    words = sentence.strip().split()
    tokenized_result = []

    for word in words:
        if word:
            current_symbols = list(word) + ['</w>']

            for rule_pair in merge_rules:
                # 查找并替换规则对
                new_symbols = []
                i = 0
                while i < len(current_symbols):
                    # 检查当前是否构成规则对
                    if i + 1 < len(current_symbols) and (current_symbols[i], current_symbols[i+1]) == rule_pair:
                        # 是则合并
                        new_symbols.append(''.join(rule_pair))
                        i += 2
                    else:
                        # 否则添加
                        new_symbols.append(current_symbols[i])
                        i += 1
                current_symbols = new_symbols
            tokenized_result.extend(current_symbols)

    return tokenized_result


cleaned_corpus = df['cleaned_text'].dropna().tolist()

if not cleaned_corpus:
    print("错误: 清洗后语料库为空")
else:
    print(f"正在 {len(cleaned_corpus)} 条清洗后的句子上训练BPE...")

    vocab, merge_rules = train_bpe(cleaned_corpus, 500)

    vocab_file_path = 'bpe_vocab.txt'
    try:
        with open(vocab_file_path, 'w', encoding='utf-8') as f:
            for token in sorted(vocab):
                f.write(token + '\n')
        print(f"词汇表已保存到 {vocab_file_path}")
    except Exception as e:
        print(f"保存词汇表时出错: {e}")

    # 对指定的示例句子进行分词
    sample_sentence = "this is a new flight"
    tokens = tokenize(sample_sentence, merge_rules)
    print(f"\n'{sample_sentence}' 的分词结果: {tokens}")


正在 14640 条清洗后的句子上训练BPE...
词汇表已保存到 bpe_vocab.txt

'this is a new flight' 的分词结果: ['th', 'i', 's</w>', 'i', 's</w>', 'a</w>', 'new</w>', 'flight</w>']


### Task 3: Feature Extraction with TF-IDF (20 Marks)

**Objective:** To represent text documents as numerical feature vectors using the TF-IDF weighting scheme.

**Description:** TF-IDF (Term Frequency-Inverse Document Frequency) weights words based on their importance in a document relative to a corpus. You will use `scikit-learn` to perform this transformation.

**Your task is to:**

1. Load the 20 Newsgroups dataset.
2. Instantiate `scikit-learn`'s `TfidfVectorizer`. Configure it with the following parameters, which is a standard practice to filter out overly common or rare words:
   * `stop_words='english'`: Use the built-in English stop words list.
   * `min_df=5`: Ignore terms that have a document frequency strictly lower than 5.
   * `max_df=0.75`: Ignore terms that have a document frequency strictly higher than 0.75 (i.e., appear in more than 75% of the documents).
3. Use the configured vectorizer to transform the text documents into a set of TF-IDF feature vectors.
4. Print the total number of documents processed and the size of the resulting vocabulary (i.e., the number of unique features).
5. Write code to identify and print the top 10 terms with the highest TF-IDF scores for the first document in the dataset.

**Dataset:**

* **The 20 Newsgroups Dataset:** A classic dataset for text applications.
* **Access:** It can be easily fetched using `sklearn.datasets.fetch_20newsgroups`.

In [ ]:
# Your code for Task 3 here
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

try:
    newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
    print("20 Newsgroups 数据集加载成功")
except Exception as e:
    print(f"加载数据集时出错: {e}")
    exit()

vectorizer = TfidfVectorizer(
    stop_words='english',
    min_df=5,
    max_df=0.75
)

try:
    tfidf_matrix = vectorizer.fit_transform(newsgroups_train.data)
    print("TF-IDF矩阵转换完成")
except Exception as e:
    print(f"转换TF-IDF矩阵时出错: {e}")
    exit()

num_documents = tfidf_matrix.shape[0]
vocab_size = tfidf_matrix.shape[1]

print(f"处理的文档总数: {num_documents}")
print(f"生成的词汇表大小: {vocab_size}")

feature_names = vectorizer.get_feature_names_out()

first_doc_tfidf = tfidf_matrix[0]
first_doc_array = first_doc_tfidf.toarray().flatten()
top_indices = np.argsort(first_doc_array)[::-1][:10]
top_terms = [(feature_names[i], first_doc_array[i]) for i in top_indices if first_doc_array[i] > 0]

print(f"\n第一篇文档的前10个TF-IDF分数最高的词:")
for term, score in top_terms:
    print(f"  {term}: {score:.4f}")


20 Newsgroups 数据集加载成功
TF-IDF矩阵转换完成
处理的文档总数: 11314
生成的词汇表大小: 17797

第一篇文档的前10个TF-IDF分数最高的词:
  car: 0.4987
  funky: 0.2404
  60s: 0.2227
  enlighten: 0.2206
  70s: 0.2206
  bumper: 0.2103
  doors: 0.1852
  sports: 0.1753
  specs: 0.1748
  production: 0.1728


: 

### Task 4: Training Word Embeddings with Gensim's Word2Vec (25 Marks)

**Objective:** To train a Word2Vec model using the `gensim` library and explore the learned semantic representations.

**Description:** Instead of building a neural network from scratch, this task uses `gensim`, a popular and efficient library for topic modeling and vector space modeling, to train word embeddings.

**Your task is to:**

1. Create a function `train_word2vec(corpus)` that takes the Brown Corpus as input.
2. Inside the function, preprocess the corpus: tokenize the text into a list of sentences, where each sentence is a list of words.
3. Instantiate and train a `gensim.models.Word2Vec` model on the preprocessed corpus. You can start with parameters like `vector_size=100`, `window=5`, and `min_count=5`.
4. The function should return the trained model.
5. Call this function to train your model.
6. After training, demonstrate your model's understanding of semantics. For each of the words "man", "love", and "house", use the trained model's methods to find and print the 5 most semantically similar words from the vocabulary.

**Dataset:**

* **The Brown Corpus:** A well-known and balanced corpus of American English text.
* **Access:** It is available directly through the NLTK library (`nltk.corpus.brown`). Use the first 50,000 sentences for manageable training time.
* **Sample Code:**
  ```python
  import nltk
  from nltk.corpus import brown
  
  # Download the Brown Corpus (if not already downloaded)
  try:
      nltk.data.find('corpora/brown')
  except nltk.downloader.DownloadError:
      nltk.download('brown')
  
  # Load the first 50,000 sentences
  corpus_sents = brown.sents()[:50000]
  print(f"Loaded {len(corpus_sents)} sentences.")
  print("Sample sentence:", corpus_sents[0])
  ```

In [ ]:
# Your code for Task 4 here
import nltk
from nltk.corpus import brown
import gensim
from gensim.models import Word2Vec

try:
    nltk.data.find('corpora/brown')
except LookupError:
    print("正在下载Brown语料库...")
    nltk.download('brown')

corpus_sents = brown.sents()[:50000]
print(f"已加载 {len(corpus_sents)} 个句子用于训练")
print(f"示例句子: {' '.join(corpus_sents[0])}")

def train_word2vec(corpus):
    # 使用Gensim的Word2Vec类进行训练
    model = Word2Vec(
        sentences=corpus, # 输入语料
        vector_size=100,  # 词向量维度
        window=5,         # 上下文窗口大小
        min_count=5,      # 最小词频阈值
        workers=4,        # 并行处理核心数
        sg=0              # sg=0 使用CBOW, sg=1 使用Skip-gram
    )

    model.build_vocab(corpus, progress_per=10000)
    model.train(corpus, total_examples=model.corpus_count, epochs=model.epochs)
    return model

print("开始训练Word2Vec模型...")
trained_model = train_word2vec(corpus_sents)
print("Word2Vec模型训练完成")

target_words = ["man", "love", "house"]

for word in target_words:
    try:
        similar_words = trained_model.wv.most_similar(word, topn=5)
        print(f"\n与 '{word}' 最相似的5个词:")
        for sim_word, similarity in similar_words:
            print(f"  {sim_word}: {similarity:.4f}")
    except KeyError:
        print(f"\n警告: 词 '{word}' 未在词汇表中找到")


已加载 50000 个句子用于训练
示例句子: The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
开始训练Word2Vec模型...


### Task 5: Intrinsic Evaluation of Your Word2Vec Embeddings (15 Marks)

**Objective:** To quantitatively evaluate the quality of your trained word embeddings by comparing their similarity scores against human judgments.

**Description:** This task measures how well the cosine similarity between your word vectors correlates with human-judged similarity scores on a benchmark dataset.

**Your task is to:**

1. Implement a function `evaluate_embeddings(model, evaluation_dataset_path)` that takes your trained `gensim` model and the path to the WordSim-353 dataset as input.
2. Inside the function, load the WordSim-353 dataset.
3. For each word pair in the dataset, calculate the cosine similarity using the word vectors from your trained model. Skip any pairs where one or both words are not present in your model's vocabulary.
4. Create two lists: one containing the human similarity scores and another containing your model's calculated cosine similarity scores for the valid pairs.
5. Calculate and print the Spearman correlation coefficient between these two lists of scores.
6. Call this function with your trained model to perform the evaluation.
7. In a markdown cell, briefly discuss your result. What does the correlation score tell you about the quality of your trained embeddings?

**Dataset:**

* **WordSim-353:** A standard dataset for evaluating semantic similarity.
* **Access:** You can find the dataset here: [WordSim-353 Test Collection](https://gabrilovich.com/resources/data/wordsim353/wordsim353.html)

In [ ]:
# Your code for Task 5 here
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

def evaluate_embeddings(model, evaluation_dataset_path):
    try:
        df_eval = pd.read_csv(evaluation_dataset_path, sep=',')
        print(f"WordSim-353 数据集 ({evaluation_dataset_path}) 加载成功：共 {len(df_eval)} 个词对")

        expected_cols = ['Word 1', 'Word 2', 'Human (mean)']
        if all(col in df_eval.columns for col in expected_cols):
            word1_col, word2_col, score_col = 'Word 1', 'Word 2', 'Human (mean)'
            print(f"识别列名: {word1_col}, {word2_col}, {score_col}")
        else:
            # 如果标准列名不存在，打印错误和实际列名
            print("错误：无法识别列名 'Word 1', 'Word 2', 'Human (mean)'")
            print("实际列名:", df_eval.columns.tolist())
            return
                
    except FileNotFoundError:
        print(f"错误：找不到评估数据集文件 '{evaluation_dataset_path}'")
        return
    except pd.errors.EmptyDataError:
        print(f"错误：文件 '{evaluation_dataset_path}' 为空")
        return
    except Exception as e:
        print(f"加载评估数据集时出错: {e}")
        return

    # 存储人工评分和模型评分
    human_scores = []
    model_scores = []

    # 遍历数据集中的每个词对
    for _, row in df_eval.iterrows():
        # 获取词对和人工评分
        word1 = row[word1_col].lower() # 转换为小写匹配模型词汇
        word2 = row[word2_col].lower()
        human_score = row[score_col]

        # 检查词是否在模型词汇表中
        if word1 in model.wv and word2 in model.wv:
            try:
                # 计算模型预测的相似度
                model_sim = model.wv.similarity(word1, word2)
                # 添加到列表
                human_scores.append(human_score)
                model_scores.append(model_sim)
                # print(f"  有效词对: {word1} - {word2}, 人工: {human_score:.2f}, 模型: {model_sim:.2f}") # 可选：打印有效词对
            except KeyError:
                # 如果仍有KeyError，跳过
                print(f"  警告：计算 {word1}-{word2} 相似度时出错")
                continue
        # 如果任一词不在词汇表，跳过该词对
        else:
            # print(f"  跳过词对（OOV）: {word1} - {word2}") # 可选：打印跳过的词对
            continue

    if len(human_scores) == 0:
        print("错误：没有找到模型词汇表中的可用词对")
        return

    correlation, p_value = spearmanr(human_scores, model_scores)

    print(f"\n评估结果:")
    print(f"有效词对数量: {len(human_scores)}")
    print(f"Spearman 相关系数: {correlation:.4f}")
    print(f"p-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print("相关性显著 (p < 0.05)")
    else:
        print("相关性不显著 (p >= 0.05)")

wordsim_path = 'combined.csv'
evaluate_embeddings(trained_model, wordsim_path)


WordSim-353 数据集 (combined.csv) 加载成功：共 353 个词对
识别列名: Word 1, Word 2, Human (mean)

评估结果:
有效词对数量: 246
Spearman 相关系数: 0.1255
p-value: 0.0492
相关性显著 (p < 0.05)


The model's performance on the WordSim-353 test was weak. The Spearman correlation coefficient was only 0.1255. This means the similarity scores predicted by the model did not match the human judgments very well. Although the result was statistically significant (p < 0.05), the low correlation suggests the trained Word2Vec model did not effectively capture the semantic relationships that humans recognize in this specific dataset. The model might need different training parameters or a larger, more suitable dataset to improve.